<a href="https://colab.research.google.com/github/timfitz04/Business-Analytics-Dissertation/blob/main/Cavan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://cavangaa.ie/fixtures-results/

In this file Cavan league tables were scraped and collected then stored.

Team names were then cleaned so the dataset would  merge with gaapitchfinder dataset adding longitudal and latiudal data to the orignial data that was scraped. Teams that did not match after cleaning were mannually matched.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
os.chdir("/content/drive/MyDrive/Dissertation")
!pwd

/content/drive/MyDrive/Dissertation


In [ ]:
!pip install bs4

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import time
time.sleep(5)   # 1 second delay between requests


compIDs_by_year = {
    2024: {
        "A-League Division 1": 208975,
        "A-League Division 2": 208977,
        "A-League Division 3": 208979,
        "B-League Division 1": 211633,
        "B-League Division 2": 211635,
        "B-League Division 3": 211637},
    2023: {
        "A-League Division 1": 190891,
        "A-League Division 2": 190893,
        "A-League Division 3": 190895,
        "B-League Division 1": 191929,
        "B-League Division 2": 191931,
        "B-League Division 3": 191933
    },
    2022: {
        "A-League Division 1": 168197,
        "A-League Division 2": 168199,
        "A-League Division 3": 168201,
        "B-League Division 1": 171915,
        "B-League Division 2": 171917,
        "B-League Division 3": 171919
    },

    2021: {
        "A-League Division 1 A": 153675,
        "A-League Division 1 B": 153677,
        "A-League Division 2 A": 153679,
        "A-League Division 2 B": 153681,
        "A-League Division 3 A": 153683,
        "A-League Division 3 B": 153685,
        "B-League Division 1 A": 156963,
        "B-League Division 1 B": 156965,
        "B-League Division 2": 156967,
        "B-League Division 3 A": 156969,
        "B-League Division 3 B": 156971},
    2019: {
        "A-League Division 1": 118861,
        "A-League Division 2": 118863,
        "A-League Division 3": 118865,
        "B-League Division 1 A": 122535,
        "B-League Division 1 B": 122537,
        "B-League Division 2": 122539,
        "B-League Division 3 A": 122541,
        "B-League Division 3 B": 122543},
    2018: {
        "A-League Division 1": 102343,
        "A-League Division 2": 102345,
        "A-League Division 3": 102347,
        "B-League Division 1": 103155,
        "B-League Division 2": 103157,
        "B-League Division 3 A": 103159,
        "B-League Division 3 B": 103161}
}

all_rows = []

for year, divisions in compIDs_by_year.items():

    for division_name, compid in divisions.items():

        url = (
            "https://www.kerrygaa.ie/fixtures-results/fixtures-results-ajax/"
            f"?countyBoardID=13&ccAC=1&compID={compid}"
            "&leagueTable=Y&resultsOnly=Y&reverseDateOrder=Y&orderTBCLast=Y"
        )

        response = requests.get(url)
        time.sleep(1)
        soup = BeautifulSoup(response.text, "html.parser")


        tables = pd.read_html(str(soup))
        if len(tables) == 0:
            print(f"No table for {division_name} {year}")
            continue

        df = tables[0]



        df.columns = (
            df.columns
            .str.strip()
            .str.lower()
            .str.replace(" ", "_")
        )

        rename_map = {
            "p": "pld",
            "team": "team",
            "w": "w",
            "d": "d",
            "l": "l",
            "f": "pf",
            "a": "pa",
            "pts": "pts"
        }
        df = df.rename(columns=rename_map)


        # Calculate points difference
        df["pd"] = df["pf"].astype(int) - df["pa"].astype(int)

        # Add division + year
        df["Division"] = division_name
        df["Year"] = year
        df["County"] = "Cavan"

        df = df[["County", "Division", "Year", "team", "pld", "w", "d", "l", "pf", "pa", "pd", "pts"]]

        all_rows.append(df)


# ---- FINAL OUTPUT ----
Cavan = pd.concat(all_rows, ignore_index=True)

print(Cavan)


/tmp/ipykernel_367/4124597931.py:82: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_367/4124597931.py:82: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_367/4124597931.py:82: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_367/4124597931.py:82: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))
/tmp/ipykernel_367/4124597931.py:82:

    County               Division  Year                        team  pld   w  \
0    Cavan    A-League Division 1  2024                Ramor United   12  11   
1    Cavan    A-League Division 1  2024                 Cavan Gaels   12  10   
2    Cavan    A-League Division 1  2024                       Gowna   12   8   
3    Cavan    A-League Division 1  2024                Crosserlough   12   8   
4    Cavan    A-League Division 1  2024            Kingscourt Stars   12   6   
..     ...                    ...   ...                         ...  ...  ..   
442  Cavan  B-League Division 3 B  2018                    Drumlane    7   4   
443  Cavan  B-League Division 3 B  2018  Templeport St. Aidan's GAA    7   4   
444  Cavan  B-League Division 3 B  2018                Killeshandra    7   3   
445  Cavan  B-League Division 3 B  2018   St Mary's GAA, Swanlinbar    7   1   
446  Cavan  B-League Division 3 B  2018               Shannon Gaels    7   0   

     d  l   pf   pa   pd  pts  
0    0 

/tmp/ipykernel_367/4124597931.py:82: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(str(soup))


In [ ]:
import re
def clean_club_name(name):
    name = name.lower()

    # remove trailing numbers (Thomas Davis 2 → Thomas Davis)
    name = re.sub(r"\s*\d+$", "", name)

    # remove reserve/second team indicators e.g. "2nd", "B", "C"
    name = re.sub(r"\b(2nd|second|team|b|c)\b", "", name, flags=re.IGNORECASE)

    # strip GAA-specific suffixes and organisation labels
    name = re.sub(r"\b(gaa|clg|club|lgfa|ladies)\b", "", name, flags=re.IGNORECASE)

    # remove punctuation + multiple spaces
    name = re.sub(r"[^\w\s]", "", name)
    name = " ".join(name.split())

    return name.strip()

# after scraping into Dub dataframe:
Cavan["clean_team"] = Cavan["team"].apply(clean_club_name)

In [ ]:
coords = pd.read_csv("/content/drive/MyDrive/Dissertation/Dub/gaapitchfinder_data.csv")
coords["clean_team"] = coords["Club"].apply(clean_club_name)
coords = coords[coords["County"] == "Cavan"].copy()
coords = coords.drop_duplicates(
    subset="clean_team",
    keep="first"
)
coords = coords[["clean_team", "Latitude", "Longitude"]]

print(coords)

                   clean_team   Latitude  Longitude
224                     cavan  53.981623  -7.360618
1379  bailieborough shamrocks  53.925765  -6.969827
1380                ballinagh  53.930673  -7.412008
1381               ballyhaise  54.046253  -7.314709
1382             ballymachugh  53.835841  -7.353148
1383   belturbet rory omoores  54.099878  -7.438112
1384            butlersbridge  54.047954  -7.375035
1385              castlerahan  53.861232  -7.209929
1386              cavan gaels  53.988038  -7.363122
1387         cootehill celtic  54.072479  -7.079643
1388                 corlough  54.118056  -7.747799
1389                cornafean  53.950400  -7.494610
1390             crosserlough  53.861975  -7.316331
1391              cuchulainns  53.808693  -6.938897
1392                     denn  53.920235  -7.264607
1393                 drumalee  53.997988  -7.352844
1394         drumgoon éire óg  54.042940  -7.018316
1395                 drumlane  54.068624  -7.478328
1396        

In [ ]:
manual_map = {
    "laragh utd": "laragh united",
    "killygarry coill an gharraí": "killygarry",
    "butlersbridge gaa": "butlersbridge",
    "templeport st aidans": "st aidans templeport",
    "searcógshercock": "shercock",
    "bailieborough": "bailieborough shamrocks",
    "kildallan gaa": "kildallan",
    "mullahoran gfc": "mullahoran dreadnoughts",
    "arva": "st patricks arva",
    "cuchulainn": "cuchulainns",
    "lavey": "erins own lavey",
    "drumgoon": "drumgoon éire óg",
    "drung": "drung dalcassians",
    "munterconnaught": "munterconnacht",
    "killeshandra": "killeshandra leaguers",
    "lacken": "lacken celtic"
}


Cavan["clean_team"] = Cavan["clean_team"].replace(manual_map)

In [ ]:
Cavan = Cavan.merge(coords, on="clean_team", how="left")


In [ ]:
Cavan[Cavan["Latitude"].isna()][["team","clean_team"]].drop_duplicates()

,team,clean_team


In [ ]:
Cavan.to_csv("/content/drive/MyDrive/Dissertation/Cavan/cavan_league_tables.csv", index=False)